In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [2]:
import csv
import io

# DVRS270826.txt has a quirk: whenever a field (e.g. legal_description) contains a
# comma, the exporter wraps the *entire* line in a stray leading/trailing '"'.
# Pandas then treats each such line as one big quoted field, silently corrupting
# ~36% of rows (all columns after the first come back NaN). Fix: strip the
# wrapping quote per line before parsing, then parse with quoting disabled since
# no real quote characters remain.
with open("DVRS270826.txt", encoding="utf-16") as f:
    lines = f.read().splitlines()

header, *rows = lines
expected_cols = len(header.split("|"))


def unwrap(line):
    if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
        return line[1:-1]
    return line


cleaned = [unwrap(line) for line in rows]

# Drop genuinely malformed rows (e.g. a stray "#NAME?" Excel-error row) instead
# of letting them silently misalign columns.
good_rows = [line for line in cleaned if len(line.split("|")) == expected_cols]
n_dropped = len(cleaned) - len(good_rows)
if n_dropped:
    print(f"Dropping {n_dropped} malformed row(s) that don't split into {expected_cols} fields")

csv_text = "\n".join([header] + good_rows)
df = pd.read_csv(io.StringIO(csv_text), sep="|", dtype=str, quoting=csv.QUOTE_NONE)


Dropping 1 malformed row(s) that don't split into 41 fields


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92010 entries, 0 to 92009
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   valuation_roll_number             92010 non-null  str  
 1   valuation_number_assessment       92010 non-null  str  
 2   valuation_number_suffix           14153 non-null  str  
 3   sale_date                         92010 non-null  str  
 4   district_code                     92010 non-null  str  
 5   sale_type                         92009 non-null  str  
 6   sales_group                       92010 non-null  str  
 7   sale_tenure                       92010 non-null  str  
 8   price_value_relationship          92010 non-null  str  
 9   sale_price_gross                  92010 non-null  str  
 10  sale_price_net                    92010 non-null  str  
 11  sale_price_chattels               92010 non-null  str  
 12  sale_price_other                  92010 non

In [5]:
# Sanity check: sale_price_gross/net should now be populated for (almost) every
# row, not just ~64% of them as before the quoting fix.
print(df.shape)
print(df[["sale_date", "sale_price_gross", "sale_price_net"]].isna().sum())

(92010, 41)
sale_date           0
sale_price_gross    0
sale_price_net      0
dtype: int64
